In [9]:
import os, sys, json
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

# ---- your settings ----
LOCAL_PROJECT_DIR = '/media/ehsan/MyFiles1/FraUAS/HIS Project/Code/ptbxl-patient-conditioned-ecg-forecasting'
LOCAL_DATA_ROOT   = '/media/ehsan/MyFiles1/FraUAS/HIS Project/PTB-XL'
LOCAL_CACHE_DIR   = 'data/study_v3'
LOCAL_RESULTS_DIR = 'results'

PILOT_TRAIN_PATIENTS, PILOT_VAL_PATIENTS = 300, 75
MEDIUM_TRAIN_PATIENTS, MEDIUM_VAL_PATIENTS = 1500, None   # None = all of fold 9

RUN_MODE = 'pilot'              # pilot | medium | full
EPOCHS = 60                     # pilot runs 2 internally; medium 60; full 120
BATCH_SIZE = 64
STUDY_ID = 'tft_patchtst_v1'    # new ID, so results don't overwrite your teammate's
RESUME = False
FINAL_TEST = False              # fold 10 stays reserved
ALLOW_FULL_CPU = False          # only True for a deliberate full run on CPU
ALLOW_PRIOR_HISTORY = False
TUNE_HIDDEN_128 = False

if RUN_MODE == 'full' and EPOCHS != 120:
    raise ValueError('The FULL preset requires EPOCHS=120.')

# ---- locate project ----
project = Path(LOCAL_PROJECT_DIR).resolve()
for f in ('src/pipeline.py', 'configs/config.yaml', 'requirements.txt'):
    assert (project / f).is_file(), f'Missing {f} under {project}'
os.chdir(project); sys.path.insert(0, str(project / 'src'))

# ---- device ----
import torch
from config import load_config, study_dir, write_json
from train import seed_everything
torch.set_num_threads(os.cpu_count())
print('PyTorch', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| threads:', torch.get_num_threads())
if RUN_MODE == 'full' and not ALLOW_FULL_CPU and not torch.cuda.is_available():
    raise RuntimeError('Full mode on CPU needs ALLOW_FULL_CPU=True (see the runtime estimates).')
seed_everything(42)

# ---- config ----
cfg = load_config()
cfg['study_id'] = STUDY_ID
cfg['run_mode'] = RUN_MODE
cfg['training'].update(batch_size=BATCH_SIZE, epochs=60 if RUN_MODE == 'full' else EPOCHS,
                       full_epochs=120, allow_full_cpu=ALLOW_FULL_CPU,
                       tune_hidden_128=TUNE_HIDDEN_128, device='cpu')
cfg['features']['prior_history'] = ALLOW_PRIOR_HISTORY
cfg['evaluation']['final_test'] = False
cfg['cohorts']['pilot']  = {'train': PILOT_TRAIN_PATIENTS,  'val': PILOT_VAL_PATIENTS}
cfg['cohorts']['medium'] = {'train': MEDIUM_TRAIN_PATIENTS, 'val': MEDIUM_VAL_PATIENTS}
cfg['paths']['results'] = str(Path(LOCAL_RESULTS_DIR).expanduser())
cfg['paths']['cache']   = str(Path(LOCAL_CACHE_DIR).expanduser())
cfg['data']['root']     = str(Path(LOCAL_DATA_ROOT).expanduser())

for req in ['ptbxl_database.csv', 'scp_statements.csv', 'SHA256SUMS.txt', 'records100']:
    assert (Path(cfg['data']['root']) / req).exists(), f'Missing dataset entry {req}'
write_json(study_dir(cfg) / 'notebook_configuration.json', cfg)
print('Results folder:', study_dir(cfg).resolve())
print('history/horizon:', cfg['data']['history'], cfg['data']['horizon'])

AssertionError: Missing src/pipeline.py under /media/ehsan/MyFiles1/FraUAS/HIS Project/Code/ptbxl-patient-conditioned-ecg-forecasting